In [1]:
import re

import pandas as pd

def extract_year(value):
    if pd.isna(value):
        return None
    match = re.search(r'\b(\d{4})\b', str(value))
    return int(match.group(1)) if match else None


In [2]:
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import JsonOutputParser


def get_census_industry(student_record: pd.Series, occupations: list[str]) -> dict:
    llm = ChatDartmouthCloud(
        model_name="vertex_ai.gemini-3-flash-preview",
        max_tokens=1024,
    )
    occupation_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to standardize a collection of industry titles to a fixed, pre-defined set "
                "of occupation titles. "
                "You will receive a record of student employment that already includes an industry title. "
                "Map this title to the best-fitting one from the provided list of allowed titles. "
                "Discuss the provided data before responding with your "
                "final decision with a valid JSON with the following keys:\n"
                "- assessment\n"
                "- occupation\n"
                "If none of the provided options are a good fit, label it as N/A."
                "The available occupation titles are:\n\n{{occupation_titles}}.",
            ),
            ("human", "Here is the employment record: \n\n {{record}}"),
        ],
        template_format="jinja2",
    )
    occupation_mapper = occupation_prompt | llm | JsonOutputParser()

    return occupation_mapper.invoke(
        input={
            "occupation_titles": occupations,
            "record": student_record.to_json(),
        }
    )

In [3]:
import json
from pathlib import Path
from joblib import Parallel, delayed
from tqdm.auto import tqdm


def process_single_record(idx, record, id_name, results_dir, occupation_year_map):
    result_file = Path(results_dir) / f"result_{idx}.json"

    if result_file.exists():
        print(f"Skipping {idx} (already processed)", end="\r")
        return {"idx": idx, "status": "skipped"}

    try:
        year = str(int(record["year"]))
        occupations = occupation_year_map.get(year, occupation_year_map["2024"])  # fallback
        response = get_census_industry(record, occupations)
        response[id_name] = record[id_name]

        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {"idx": idx, "id": record[id_name], "status": "success", "result": response}

    except Exception as e:
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "id": record[id_name], "error": str(e)}, f, indent=2)
        return {"idx": idx, "id": record[id_name], "status": "failed", "error": str(e)}


def process_parallel_with_saves(records, id_name, occupation_year_map, results_dir="results/occupations"):
    Path(results_dir).mkdir(parents=True, exist_ok=True)
    records_data = [(idx, row) for idx, row in records.iterrows()]

    results = Parallel(n_jobs=-1)(
        delayed(process_single_record)(idx, record, id_name, results_dir, occupation_year_map)
        for idx, record in tqdm(records_data, desc="Processing records")
    )

    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(" " * 100, end="\r")
    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results

In [37]:
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm


class LLMBatchProcessor:
    def __init__(self, records, id_name, occupation_year_map, process_fn,
                 results_dir="results", n_workers=8):
        self.records = records
        self.id_name = id_name
        self.occupation_year_map = occupation_year_map
        self.process_fn = process_fn  # e.g. get_census_occupation or get_census_industry
        self.results_dir = Path(results_dir)
        self.n_workers = n_workers
        self.results_dir.mkdir(parents=True, exist_ok=True)

    def _process_single(self, idx, record):
        result_file = self.results_dir / f"result_{idx}.json"
        if result_file.exists():
            return {"idx": idx, "status": "skipped"}

        try:
            year = str(int(record["year"]))
            occupations = self.occupation_year_map.get(year, self.occupation_year_map["2024"])
            response = self.process_fn(record, occupations)
            response[self.id_name] = record[self.id_name]

            with open(result_file, "w") as f:
                json.dump({"idx": idx, "result": response}, f, indent=2)

            return {"idx": idx, "id": record[self.id_name], "status": "success"}

        except Exception as e:
            error_file = self.results_dir / f"error_{idx}.json"
            with open(error_file, "w") as f:
                json.dump({"idx": idx, "id": record[self.id_name], "error": str(e)}, f, indent=2)
            return {"idx": idx, "id": record[self.id_name], "status": "failed", "error": str(e)}

    def _load_done_ids(self):
        done_ids = set()
        for f in self.results_dir.glob("result_*.json"):
            try:
                result = json.load(open(f))["result"]
                done_ids.add(result.get(self.id_name) or result.get("id"))
            except (KeyError, json.JSONDecodeError):
                pass
        return done_ids

    def _run_batch(self, records_df):
        records_data = list(records_df.iterrows())
        batch_results = []

        with ThreadPoolExecutor(max_workers=self.n_workers) as pool:
            futures = {
                pool.submit(self._process_single, idx, record): idx
                for idx, record in records_data
            }
            for f in tqdm(as_completed(futures), total=len(futures), desc="Processing"):
                batch_results.append(f.result())

        return batch_results

    def _print_summary(self, batch_results):
        counts = {"success": 0, "failed": 0, "skipped": 0}
        for r in batch_results:
            counts[r["status"]] += 1
        print(f"✅ {counts['success']}  ❌ {counts['failed']}  ⏭️ {counts['skipped']}")

    def process(self):
        batch = self._run_batch(self.records)
        self._print_summary(batch)
        return self

    def retry(self, max_retries=7):
        for attempt in range(max_retries):
            done_ids = self._load_done_ids()
            remaining_df = self.records[~self.records[self.id_name].isin(done_ids)]

            if remaining_df.empty:
                print("All records processed successfully!")
                break

            for ef in self.results_dir.glob("error_*.json"):
                ef.unlink()

            print(f"\nRetry {attempt + 1}/{max_retries} — {len(remaining_df)} remaining")
            batch = self._run_batch(remaining_df)
            self._print_summary(batch)
        else:
            remaining = len(self.records) - len(self._load_done_ids())
            if remaining:
                print(f"\nMax retries reached. {remaining} records remain.")

        return self

    def collect(self):
        records = []
        for file in sorted(self.results_dir.glob("result_*.json")):
            try:
                records.append(json.load(open(file))["result"])
            except (KeyError, json.JSONDecodeError):
                print(f"Malformed: {file.name}")
        return records

In [5]:
import pandas as pd


admissions = pd.read_csv("../data/derived/admissions.csv")
admissions_relevant_columns = [
    "Student ID",
    "Job 1 Organization",
    "Job #1 Industry Code",
    "Job 1 Title",
    "Round",
    "year"
]
admissions['year'] = admissions['Round'].apply(extract_year)

In [6]:
interviews = pd.read_csv("../data/derived/interviews.csv")
interviews_relevant_columns = [
    "Student ID",
    "Employer",
    "Industry",
    "Job Function",
    "Application Date",
    "year",
]

interviews['year'] = interviews['Application Date'].apply(extract_year)

In [31]:
outcomes = pd.read_csv("../data/derived/outcomes.csv")
outcomes_relevant_columns = [
    # "Student ID",
    "Outcome ID",
    "Employer",
    "Detailed Function",
    "Detailed Industry",
    "Reported Date",
    "Offer Received Date",
    "year",
]
outcomes['year'] = outcomes['Offer Received Date'].apply(extract_year)
outcomes.loc[outcomes['year'] == 1923, 'year'] = 2023
outcomes.loc[outcomes['year'] == 1918, 'year'] = 2018

In [8]:
industry_year_map = {}

for year in range(2015, 2025):
    with open(f"../data/industry_{year}.txt", 'r') as f:
        industry_year_map[str(year)] = f.read().splitlines()

In [12]:
results_dir = "results/industry/admissions"

In [ ]:
admissions_processor = LLMBatchProcessor(
        admissions[admissions_relevant_columns],
        id_name="Student ID",
        occupation_year_map=industry_year_map,
        process_fn=get_census_industry,
        results_dir=results_dir,
    )

In [ ]:
admissions_processor.process().retry()


Retry 1/7 — 2 records remaining


Processing records:   0%|          | 0/2 [00:00<?, ?it/s]

Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /var/folders/rt/xv7r5qdj52742frmp1y_56940000gp/T/joblib_memmapping_folder_27278_8fb3f80be89b4efe97ae9c89b301ad35_c599e9a34db84fceb1c4da0d5240e80c for automatic cleanup: unknown resource type folder
Traceback (most recent call last):
  File "/Users/f006p17/.local/share/uv/python/cpython-3.13.12-macos-aarch64-none/lib/python3.13/multiprocessing/resource_tracker.py", line 371, in main
    raise ValueError(
        f'Cannot register {name} for automatic cleanup: '
        f'unknown resource type {rtype}')
ValueError: Cannot register /loky-27278-0qqr5b6m for automatic cleanup: unknown resource type semlock
Traceback (most recent call last):
  File "/Users/f006

✅ Successful: 2  ❌ Failed: 0  ⏭️ Skipped: 0
All records processed successfully!


In [16]:
records = admissions_processor.collect()

In [27]:
admissions_industry_df = pd.DataFrame.from_records(records)
admissions_industry_df.rename(columns={'Student ID': 'student_id'}, inplace=True)

In [29]:
admissions_industry_df.to_csv("../data/derived/admissions_industry_mapping.csv", index=False)

In [38]:

outcomes_filtered = outcomes[outcomes['year'].notna()]

In [39]:
outcomes_processor = LLMBatchProcessor(
        outcomes_filtered[outcomes_relevant_columns],
        id_name="Outcome ID",
        occupation_year_map=industry_year_map,
        results_dir="results/industry/outcomes",
        process_fn=get_census_industry,
    )

In [40]:
outcomes_records = outcomes_processor.process().retry().collect()

Processing:   0%|          | 0/5063 [00:00<?, ?it/s]

✅ 5063  ❌ 0  ⏭️ 0
All records processed successfully!


In [42]:
outcomes_df = pd.DataFrame.from_records(outcomes_records)
outcomes_df.rename(columns={"Outcome ID": "outcome_id"}, inplace=True)

In [43]:
outcomes_df.to_csv("../data/derived/outcomes_industry_mapping.csv", index=False)

In [53]:
interviews_processor = LLMBatchProcessor(
        interviews[interviews_relevant_columns],
        id_name="Student ID",
        occupation_year_map=industry_year_map,
        results_dir="results/industry/interviews",
        process_fn=get_census_industry,
    )

In [54]:
interviews_records = interviews_processor.process().retry().collect()

Processing:   0%|          | 0/6876 [00:00<?, ?it/s]

✅ 6876  ❌ 0  ⏭️ 0
All records processed successfully!


In [64]:
import pandas as pd
import json
from pathlib import Path

# Read all JSON files from directory
data = []
for f in Path("./results/industry/interviews").glob("*.json"):
    with open(f) as fp:
        data.append(json.load(fp)) 

In [68]:
df = pd.DataFrame([{**item['result'], 'idx': item['idx']} for item in data])

In [71]:
interviews = (
    interviews.reset_index(names="idx")
    .merge(
        right=df.rename(columns={"occupation": "census_industry"}),
        on=["idx", "Student ID"],
    )
    .drop(columns=["assessment", "idx"])
)

In [74]:
interviews.to_csv("../data/derived/interviews_industry_mapping.csv", index=False)